# Kashubian audio

> "Basic exploration for a scraper"

- branch: master
- hidden: true
- categories: [scraper, kashubian]

Based on [this notebook]({% post_url 2025-06-15-kashubian-audio %})

In [1]:
URL = "http://web.archive.org/web/20180625150625/http://www.miesiecznikpomerania.pl/audio"

In [2]:
import requests
from bs4 import BeautifulSoup
import json

In [3]:
csb_audio_page = requests.get(URL)
assert csb_audio_page.status_code == 200, "Failed to fetch the page"
soup = BeautifulSoup(csb_audio_page.content, "html.parser")

In [21]:
segments = []

for accordion in soup.find_all("div", class_="sp-accordion-inner"):
    data = {}
    paragraphs = []
    # walk through the children for each accordion and extract the text from <p> and <strong> tags within a bare <div> tag
    for child in accordion.children:
        if child.name == "div" and not child.attrs:
            for grandchild in child.children:
                if grandchild.name in ["p", "strong", "span"]:
                    # check if the element contains <audio>; if so, extract <source src> to data["audio"]
                    audio_tag = grandchild.find("audio")
                    if audio_tag:
                        source_tag = audio_tag.find("source")
                        if source_tag and source_tag.has_attr("src"):
                            data["audio"] = source_tag["src"]
                            continue  # skip adding text if audio is present
                    text_content = grandchild.get_text(strip=True)
                    if text_content:
                        paragraphs.append(text_content)
    data["text"] = paragraphs
    segments.append(data)

In [25]:
import json

with open("csb_audio_segments.json", "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=4)